# Ch11 — 卡方檢定 (Chi-Squared Tests)

> **預計時間：1.5 小時**  
> **資料集：** `datasets/titanic_train.csv`  
> **前置章節：** Ch07 假設檢定基礎

## 學習目標

完成本章後，你將能夠：

1. 理解卡方統計量 (Chi-Squared Statistic) 的計算原理
2. 執行**適合度檢定** (Goodness-of-Fit Test) 判斷類別分佈是否符合預期
3. 執行**獨立性檢定** (Test of Independence) 判斷兩個類別變數是否相關
4. 知道何時改用 **Fisher's Exact Test**
5. 計算效果量 **Cramer's V** 以量化關聯強度
6. 將卡方檢定應用於**特徵選擇** (Feature Selection)

---
## 環境設定

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# 顯示設定
plt.rcParams["font.size"] = 12
plt.rcParams["figure.figsize"] = (8, 5)
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 15)

print("環境載入完成")

In [ ]:
# 載入 Titanic 資料集
df = pd.read_csv("datasets/titanic_train.csv")
print(f"資料集大小: {df.shape}")
df.head()

In [ ]:
# 快速檢視類別變數
cat_cols = ["Survived", "Pclass", "Sex", "Embarked", "SibSp", "Parch"]
df[cat_cols].describe()

---
## Part A — 卡方檢定的概念

### 觀察值 (Observed) vs 期望值 (Expected)

卡方檢定的核心概念非常直覺：

- **觀察值 (Observed, O)**：實際在資料中看到的次數
- **期望值 (Expected, E)**：如果虛無假設成立，理論上應該出現的次數

如果觀察值和期望值差距很大，就有理由懷疑虛無假設不成立。

### 卡方統計量 (Chi-Squared Statistic)

$$\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}$$

- 每個類別的差距取平方後，除以期望值做標準化
- 加總所有類別的結果，得到一個總體偏離程度
- $\chi^2$ 值越大，代表觀察值偏離期望值越多

### 自由度 (Degrees of Freedom, df)

| 檢定類型 | 自由度公式 |
|:---|:---|
| 適合度檢定 (Goodness-of-Fit) | $df = k - 1$（k = 類別數） |
| 獨立性檢定 (Test of Independence) | $df = (r-1) \times (c-1)$（r = 列數, c = 欄數） |

自由度決定了卡方分佈 (Chi-Squared Distribution) 的形狀，進而影響 p-value 的計算。

In [ ]:
# 視覺化不同自由度的卡方分佈
x = np.linspace(0, 20, 300)
fig, ax = plt.subplots(figsize=(9, 5))

for d in [1, 2, 4, 6, 10]:
    ax.plot(x, stats.chi2.pdf(x, df=d), label=f"df = {d}")

ax.set_xlabel("$\\chi^2$")
ax.set_ylabel("機率密度 (Density)")
ax.set_title("卡方分佈 (Chi-Squared Distribution) — 不同自由度")
ax.legend()
ax.set_ylim(0, 0.5)
plt.tight_layout()
plt.show()

---
## Part B — 適合度檢定 (Goodness-of-Fit Test)

### 問題：Titanic 各登船港口 (Embarked) 的乘客比例是否均等？

- **H0（虛無假設）：** 三個港口（S, C, Q）的乘客比例相同（各占 1/3）
- **H1（對立假設）：** 至少有一個港口的比例不同
- **顯著水準：** $\alpha = 0.05$

In [ ]:
# 觀察各港口的實際人數
embarked_counts = df["Embarked"].value_counts().sort_index()
print("觀察值 (Observed):")
print(embarked_counts)
print(f"\n總人數: {embarked_counts.sum()}")

In [ ]:
# 計算期望值（假設均等分佈）
n = embarked_counts.sum()
k = len(embarked_counts)
expected = np.array([n / k] * k)

print(f"期望值 (Expected): 每個港口 {n/k:.1f} 人")
print(f"自由度 (df): k - 1 = {k} - 1 = {k - 1}")

In [ ]:
# 手動計算卡方統計量
observed = embarked_counts.values
chi2_manual = np.sum((observed - expected) ** 2 / expected)
print(f"手動計算 chi2 = {chi2_manual:.4f}")

In [ ]:
# 使用 scipy 進行適合度檢定
chi2_stat, p_value = stats.chisquare(f_obs=observed, f_exp=expected)

print("=" * 50)
print("適合度檢定 (Goodness-of-Fit Test) 結果")
print("=" * 50)
print(f"卡方統計量 (chi2): {chi2_stat:.4f}")
print(f"p-value:           {p_value:.6f}")
print(f"自由度 (df):       {k - 1}")
print(f"顯著水準 (alpha):  0.05")
print("-" * 50)

if p_value < 0.05:
    print("結論: 拒絕 H0 — 各港口的乘客比例並不均等。")
else:
    print("結論: 無法拒絕 H0 — 沒有足夠證據認為各港口比例不同。")

In [ ]:
# 視覺化：觀察值 vs 期望值
fig, ax = plt.subplots(figsize=(7, 5))

x_pos = np.arange(k)
width = 0.35

ax.bar(x_pos - width/2, observed, width, label="觀察值 (Observed)", color="steelblue")
ax.bar(x_pos + width/2, expected, width, label="期望值 (Expected)", color="coral")

ax.set_xticks(x_pos)
ax.set_xticklabels(embarked_counts.index)
ax.set_xlabel("登船港口 (Embarked)")
ax.set_ylabel("人數")
ax.set_title("適合度檢定：各港口乘客人數 — 觀察值 vs 期望值")
ax.legend()
plt.tight_layout()
plt.show()

### 解讀

p-value 極小，遠低於 0.05，因此我們**拒絕虛無假設**。

Titanic 乘客的登船港口分佈**並不均等**——Southampton (S) 的乘客數量遠多於 Cherbourg (C) 和 Queenstown (Q)。
這與歷史事實一致：Southampton 是 Titanic 的出發港，大部分乘客從這裡上船。

---
## Part C — 獨立性檢定 (Test of Independence)

### 範例 1：Sex 和 Survived 是否獨立？

- **H0：** 性別 (Sex) 和存活 (Survived) 是**獨立**的（性別不影響存活率）
- **H1：** 性別和存活**不獨立**（性別與存活率有關聯）
- **顯著水準：** $\alpha = 0.05$

In [ ]:
# 建立列聯表 (Contingency Table)
ct_sex = pd.crosstab(df["Sex"], df["Survived"], margins=True)
ct_sex.columns = ["死亡 (0)", "存活 (1)", "合計"]
ct_sex.index = ["女性 (female)", "男性 (male)", "合計"]
ct_sex

In [ ]:
# 執行卡方獨立性檢定
ct_raw = pd.crosstab(df["Sex"], df["Survived"])
chi2, p, dof, expected_freq = stats.chi2_contingency(ct_raw)

print("=" * 50)
print("獨立性檢定結果：Sex vs Survived")
print("=" * 50)
print(f"卡方統計量 (chi2):  {chi2:.4f}")
print(f"p-value:            {p:.2e}")
print(f"自由度 (df):        {dof}")
print(f"\n期望頻次矩陣 (Expected Frequencies):")
print(pd.DataFrame(
    expected_freq,
    index=ct_raw.index,
    columns=ct_raw.columns
).round(2))

In [ ]:
# 解讀結果
alpha = 0.05
print("-" * 50)
if p < alpha:
    print(f"p-value ({p:.2e}) < alpha ({alpha})")
    print("結論: 拒絕 H0 — 性別與存活率不獨立。")
    print("也就是說，性別與存活與否之間存在顯著關聯。")
else:
    print(f"p-value ({p:.4f}) >= alpha ({alpha})")
    print("結論: 無法拒絕 H0。")

In [ ]:
# 視覺化：Sex vs Survived
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左：堆疊長條圖
ct_plot = pd.crosstab(df["Sex"], df["Survived"], normalize="index") * 100
ct_plot.columns = ["死亡", "存活"]
ct_plot.plot(kind="bar", stacked=True, ax=axes[0], color=["#e74c3c", "#2ecc71"])
axes[0].set_title("性別 vs 存活率")
axes[0].set_ylabel("百分比 (%)")
axes[0].set_xlabel("性別 (Sex)")
axes[0].legend(title="結果")
axes[0].tick_params(axis="x", rotation=0)

# 右：熱力圖 (觀察值 vs 期望值的殘差)
residuals = (ct_raw.values - expected_freq) / np.sqrt(expected_freq)
sns.heatmap(
    residuals,
    annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    xticklabels=["死亡 (0)", "存活 (1)"],
    yticklabels=["female", "male"],
    ax=axes[1]
)
axes[1].set_title("標準化殘差 (Standardized Residuals)")

plt.tight_layout()
plt.show()

### 範例 2：Pclass 和 Survived 是否獨立？

- **H0：** 艙等 (Pclass) 和存活 (Survived) 是獨立的
- **H1：** 艙等和存活不獨立

In [ ]:
# 列聯表
ct_pclass = pd.crosstab(df["Pclass"], df["Survived"], margins=True)
ct_pclass.columns = ["死亡 (0)", "存活 (1)", "合計"]
ct_pclass.index = ["1st", "2nd", "3rd", "合計"]
ct_pclass

In [ ]:
# 獨立性檢定
ct_pclass_raw = pd.crosstab(df["Pclass"], df["Survived"])
chi2_p, p_p, dof_p, exp_p = stats.chi2_contingency(ct_pclass_raw)

print("=" * 50)
print("獨立性檢定結果：Pclass vs Survived")
print("=" * 50)
print(f"卡方統計量 (chi2):  {chi2_p:.4f}")
print(f"p-value:            {p_p:.2e}")
print(f"自由度 (df):        {dof_p}")
print(f"\n期望頻次矩陣:")
print(pd.DataFrame(
    exp_p,
    index=ct_pclass_raw.index,
    columns=ct_pclass_raw.columns
).round(2))
print("-" * 50)
if p_p < 0.05:
    print("結論: 拒絕 H0 — 艙等與存活率不獨立。")
else:
    print("結論: 無法拒絕 H0。")

In [ ]:
# 視覺化：Pclass vs Survived
fig, ax = plt.subplots(figsize=(8, 5))

ct_pclass_pct = pd.crosstab(df["Pclass"], df["Survived"], normalize="index") * 100
ct_pclass_pct.columns = ["死亡", "存活"]
ct_pclass_pct.plot(kind="bar", stacked=True, ax=ax, color=["#e74c3c", "#2ecc71"])

ax.set_title(f"艙等 vs 存活率 (chi2={chi2_p:.1f}, p={p_p:.2e})")
ax.set_ylabel("百分比 (%)")
ax.set_xlabel("艙等 (Pclass)")
ax.legend(title="結果")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

### 解讀

兩個檢定的 p-value 都極小：

- **Sex vs Survived：** 女性的存活率遠高於男性（"Women and children first" 政策）
- **Pclass vs Survived：** 頭等艙乘客的存活率最高，三等艙最低

這些結果都與 Titanic 的歷史記載一致。

---
## Part D — Fisher's Exact Test

### 何時使用 Fisher's Exact Test?

卡方檢定有一個重要假設：**每個格子的期望頻次至少為 5**。

當期望頻次 < 5 的格子超過 20% 時，卡方近似不可靠，應改用 **Fisher's 精確檢定** (Fisher's Exact Test)。

| 條件 | 建議方法 |
|:---|:---|
| 所有期望頻次 >= 5 | 卡方檢定 |
| 期望頻次 < 5 超過 20%，且為 2x2 表格 | Fisher's Exact Test |
| 樣本很小 (n < 20~30) | Fisher's Exact Test |

> **注意：** `scipy.stats.fisher_exact()` 僅支援 **2x2** 表格。

In [ ]:
# 建構一個小樣本情境：抽取少量乘客
np.random.seed(42)
df_small = df.dropna(subset=["Embarked"]).sample(n=20, random_state=42)

# 建立 2x2 表格：Sex vs Survived
ct_small = pd.crosstab(df_small["Sex"], df_small["Survived"])
print("小樣本列聯表 (n=20):")
print(ct_small)

In [ ]:
# 檢查期望頻次
chi2_s, p_s, dof_s, exp_s = stats.chi2_contingency(ct_small)
print("期望頻次矩陣:")
print(np.round(exp_s, 2))

cells_below_5 = (exp_s < 5).sum()
total_cells = exp_s.size
pct_below = cells_below_5 / total_cells * 100
print(f"\n期望頻次 < 5 的格子: {cells_below_5}/{total_cells} ({pct_below:.0f}%)")

if pct_below > 20:
    print("建議使用 Fisher's Exact Test!")
else:
    print("卡方檢定的假設成立，可以使用卡方檢定。")

In [ ]:
# Fisher's Exact Test (2x2)
odds_ratio, p_fisher = stats.fisher_exact(ct_small)

print("=" * 50)
print("Fisher's Exact Test 結果")
print("=" * 50)
print(f"勝算比 (Odds Ratio): {odds_ratio:.4f}")
print(f"p-value:             {p_fisher:.4f}")
print("-" * 50)

if p_fisher < 0.05:
    print("結論: 拒絕 H0 — 在這個小樣本中，性別與存活率有顯著關聯。")
else:
    print("結論: 無法拒絕 H0 — 在這個小樣本中，沒有足夠證據。")

print(f"\n比較：卡方檢定 p-value = {p_s:.4f}")
print("（小樣本下兩者結果可能不同，Fisher's Exact Test 更可靠）")

---
## Part E — 效果量：Cramer's V

### 為什麼需要效果量？

p-value 告訴我們關聯是否**顯著**，但不告訴我們關聯有多**強**。
樣本越大，即使很微弱的關聯也可能產生顯著的 p-value。

**Cramer's V** 是衡量兩個類別變數關聯強度的指標：

$$V = \sqrt{\frac{\chi^2}{n \times \min(r-1, c-1)}}$$

其中：
- $\chi^2$ = 卡方統計量
- $n$ = 總樣本數
- $r$ = 列數, $c$ = 欄數

### Cramer's V 解讀標準

| V 值 | 關聯強度 |
|:---|:---|
| 0.1 左右 | 小 (Small) |
| 0.3 左右 | 中 (Medium) |
| 0.5 以上 | 大 (Large) |

In [ ]:
def cramers_v(contingency_table):
    """計算 Cramer's V 效果量。

    Parameters
    ----------
    contingency_table : pd.DataFrame or np.ndarray
        列聯表（不含邊際合計）

    Returns
    -------
    float
        Cramer's V 值 (0 ~ 1)
    """
    chi2, _, _, _ = stats.chi2_contingency(contingency_table)
    n = np.array(contingency_table).sum()
    r, c = np.array(contingency_table).shape
    v = np.sqrt(chi2 / (n * min(r - 1, c - 1)))
    return v

In [ ]:
# 計算 Sex vs Survived 的 Cramer's V
v_sex = cramers_v(ct_raw)
print(f"Sex vs Survived:    Cramer's V = {v_sex:.4f}")

# 計算 Pclass vs Survived 的 Cramer's V
v_pclass = cramers_v(ct_pclass_raw)
print(f"Pclass vs Survived: Cramer's V = {v_pclass:.4f}")

# 解讀
def interpret_cramers_v(v):
    if v < 0.1:
        return "可忽略 (Negligible)"
    elif v < 0.3:
        return "小 (Small)"
    elif v < 0.5:
        return "中 (Medium)"
    else:
        return "大 (Large)"

print(f"\nSex 的關聯強度:    {interpret_cramers_v(v_sex)}")
print(f"Pclass 的關聯強度: {interpret_cramers_v(v_pclass)}")

---
## Part F — 實務應用：特徵與目標的關聯

### 批次檢定：所有類別特徵 vs Survived

在機器學習的特徵選擇階段，我們經常需要快速判斷哪些類別特徵與目標變數有關聯。
卡方檢定可以幫助我們系統性地篩選特徵。

In [ ]:
# 選擇要測試的類別特徵
features = ["Pclass", "Sex", "SibSp", "Parch", "Embarked"]
target = "Survived"

results = []

for feat in features:
    # 移除缺失值
    subset = df[[feat, target]].dropna()
    ct = pd.crosstab(subset[feat], subset[target])

    chi2_val, p_val, dof_val, exp_val = stats.chi2_contingency(ct)
    v = cramers_v(ct)

    # 檢查期望頻次假設
    low_exp = (exp_val < 5).sum()
    total = exp_val.size
    warning = "*" if low_exp / total > 0.2 else ""

    results.append({
        "特徵": feat,
        "chi2": round(chi2_val, 2),
        "p-value": p_val,
        "df": dof_val,
        "Cramer's V": round(v, 4),
        "顯著": "Yes" if p_val < 0.05 else "No",
        "注意": warning
    })

results_df = pd.DataFrame(results).sort_values("p-value")
results_df

In [ ]:
# Heatmap：p-value 視覺化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左：p-value（取 -log10 讓顯著的值更大）
pvals = results_df.set_index("特徵")[["p-value"]].copy()
pvals["-log10(p)"] = -np.log10(pvals["p-value"].clip(lower=1e-300))

sns.heatmap(
    pvals[["-log10(p)"]].T,
    annot=True, fmt=".1f", cmap="YlOrRd",
    ax=axes[0], cbar_kws={"label": "-log10(p-value)"}
)
axes[0].set_title("各特徵 vs Survived 的 p-value\n(值越大 = 越顯著)")
axes[0].set_ylabel("")

# 右：Cramer's V
v_vals = results_df.set_index("特徵")[["Cramer's V"]]
sns.heatmap(
    v_vals.T,
    annot=True, fmt=".3f", cmap="Blues",
    ax=axes[1], cbar_kws={"label": "Cramer's V"}
)
axes[1].set_title("各特徵 vs Survived 的 Cramer's V\n(值越大 = 關聯越強)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

### 知識補給站：卡方檢定在機器學習特徵選擇中的應用

在機器學習中，`sklearn.feature_selection.chi2` 函式可以直接用於特徵選擇：

```python
from sklearn.feature_selection import chi2, SelectKBest

# 選擇與目標最相關的 k 個特徵
selector = SelectKBest(chi2, k=5)
X_selected = selector.fit_transform(X, y)
```

**使用時機：**
- 類別型特徵（或已離散化的數值特徵）與類別型目標變數
- 高維度資料中快速篩選相關特徵
- 與互資訊 (Mutual Information) 搭配使用

**注意事項：**
- `sklearn.feature_selection.chi2` 要求**非負值**輸入
- 卡方檢定只能偵測關聯的「有無」，無法判斷方向
- 對於連續型特徵，建議使用 ANOVA F-test 或互資訊

---
## 重點整理

| 檢定方法 | 用途 | scipy 函式 | 適用條件 |
|:---|:---|:---|:---|
| 適合度檢定 (GoF) | 單一變數的分佈是否符合預期 | `stats.chisquare()` | 期望頻次 >= 5 |
| 獨立性檢定 | 兩個類別變數是否獨立 | `stats.chi2_contingency()` | 期望頻次 >= 5 |
| Fisher's Exact Test | 2x2 表格的精確檢定 | `stats.fisher_exact()` | 小樣本 / 期望頻次 < 5 |
| Cramer's V | 效果量 | 手動計算 | 搭配卡方檢定使用 |

**關鍵要點：**

1. 卡方檢定比較的是**觀察值與期望值的偏離程度**
2. 期望頻次太小時，改用 **Fisher's Exact Test**
3. p-value 說「有沒有」關聯，**Cramer's V** 說「多強」
4. 卡方檢定可用於機器學習的**特徵選擇**

---
## 練習題

### 練習 1 (Easy)

將 `Age` 分成三組（`young`, `middle`, `senior`），檢定年齡組與 `Survived` 是否獨立。

提示：使用 `pd.cut()` 將 Age 分組。

In [ ]:
# 練習 1 — 請在此撰寫你的程式碼


### 練習 2 (Medium)

檢定 `Embarked` 與 `Pclass` 是否獨立，並計算 Cramer's V。
結果是否合理？嘗試解釋為什麼。

In [ ]:
# 練習 2 — 請在此撰寫你的程式碼


### 練習 3 (Hard)

撰寫一個函式 `chi2_feature_report(df, target, features, alpha=0.05)`，接收 DataFrame、目標欄位名稱和特徵清單，回傳包含以下資訊的 DataFrame：
- 卡方統計量、p-value、df、Cramer's V
- 是否通過期望頻次假設
- 建議使用的檢定方法（卡方 / Fisher's Exact）

用 Titanic 全部類別特徵測試你的函式。

In [ ]:
# 練習 3 — 請在此撰寫你的程式碼


---

[< Ch10 — 無母數檢定](10_無母數檢定.ipynb) | [Ch12 — 相關分析與迴歸入門 >](12_相關分析與迴歸入門.ipynb)